# Lab 13 Part 1 Information Retrieval for RAG

## 1 Information Retrieval

## 2 Create the Document Collection

In [1]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents = [
    "Transformers use self-attention to understand relationships between words.",
    "TF-IDF represents documents using word importance scores.",
    "Word2Vec learns dense vector representations of words.",
    "Named entity recognition identifies names, places, and organizations in text.",
    "Information retrieval finds relevant documents for a user query.",
    "Cosine similarity measures how similar two text vectors are."
]
for i, document in enumerate(documents):
    print(f'Document {i}: {document}')

Document 0: Transformers use self-attention to understand relationships between words.
Document 1: TF-IDF represents documents using word importance scores.
Document 2: Word2Vec learns dense vector representations of words.
Document 3: Named entity recognition identifies names, places, and organizations in text.
Document 4: Information retrieval finds relevant documents for a user query.
Document 5: Cosine similarity measures how similar two text vectors are.


## 3 Convert Documents into TF IDF Vectors

In [2]:
vectorizer = TfidfVectorizer(stop_words='english')
document_vectors = vectorizer.fit_transform(documents)
print('TF-IDF matrix shape:', document_vectors.shape)
print('Vocabulary:', vectorizer.get_feature_names_out().tolist())
print('First document vector:', document_vectors[0].toarray().round(2))

TF-IDF matrix shape: (6, 39)
Vocabulary: ['attention', 'cosine', 'dense', 'documents', 'entity', 'finds', 'identifies', 'idf', 'importance', 'information', 'learns', 'measures', 'named', 'names', 'organizations', 'places', 'query', 'recognition', 'relationships', 'relevant', 'representations', 'represents', 'retrieval', 'scores', 'self', 'similar', 'similarity', 'text', 'tf', 'transformers', 'understand', 'use', 'user', 'using', 'vector', 'vectors', 'word', 'word2vec', 'words']
First document vector: [[0.39 0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
  0.   0.   0.   0.   0.39 0.   0.   0.   0.   0.   0.39 0.   0.   0.
  0.   0.39 0.39 0.39 0.   0.   0.   0.   0.   0.   0.32]]


## 4 Search Using a Query

In [3]:
query = 'How do transformers understand words?'
query_vector = vectorizer.transform([query])
scores = cosine_similarity(query_vector, document_vectors)[0]
for i, score in enumerate(scores):
    print(f'Document {i}: {score:.3f}')

Document 0: 0.633
Document 1: 0.000
Document 2: 0.173
Document 3: 0.000
Document 4: 0.000
Document 5: 0.000


## 5 Retrieve the Top k Documents

In [4]:
k = 3
ranked_ids = np.lexsort((np.arange(len(documents)), -scores))[:k]
for rank, i in enumerate(ranked_ids, start=1):
    print(f'Rank {rank} | Document {i} | Score: {scores[i]:.3f}')
    print(documents[i])
    print()

Rank 1 | Document 0 | Score: 0.633
Transformers use self-attention to understand relationships between words.

Rank 2 | Document 2 | Score: 0.173
Word2Vec learns dense vector representations of words.

Rank 3 | Document 1 | Score: 0.000
TF-IDF represents documents using word importance scores.



## Search Function

In [5]:
def search(query, k=3):
    if not isinstance(query, str):
        raise TypeError('The query must be a string.')
    if isinstance(k, bool) or not isinstance(k, (int, np.integer)) or not 1 <= k <= len(documents):
        raise ValueError(f'k must be an integer from 1 to {len(documents)}.')
    query_vector = vectorizer.transform([query])
    similarities = cosine_similarity(query_vector, document_vectors)[0]
    order = np.lexsort((np.arange(len(documents)), -similarities))[:k]
    results = [dict(rank=rank, document_id=int(index), score=float(similarities[index]), text=documents[index])
               for rank, index in enumerate(order, start=1)]
    print(f'Query: {query!r} | k={k}')
    if query_vector.nnz == 0:
        print('No query terms match the fitted vocabulary. All scores are zero; ties use ascending document ID.')
    for result in results:
        print(f"Rank {result['rank']} | Document {result['document_id']} | Score: {result['score']:.3f}")
        print(result['text'])
    return results

## 6 Student Exercises

### Exercise 1 Change the Query

In [6]:
ner_results = search('named entity recognition', k=3)
print('First-ranked document:', ner_results[0]['document_id'])

Query: 'named entity recognition' | k=3
Rank 1 | Document 3 | Score: 0.625
Named entity recognition identifies names, places, and organizations in text.
Rank 2 | Document 0 | Score: 0.000
Transformers use self-attention to understand relationships between words.
Rank 3 | Document 1 | Score: 0.000
TF-IDF represents documents using word importance scores.
First-ranked document: 3


### Exercise 1 Answer

Document **3** ranks first: it describes named entity recognition.

### Exercise 2 Change k

In [7]:
representation_query = 'document representation'
results_k1 = search(representation_query, k=1)
print()
results_k3 = search(representation_query, k=3)
print()
print('Number of results:', len(results_k1), 'versus', len(results_k3))

Query: 'document representation' | k=1
No query terms match the fitted vocabulary. All scores are zero; ties use ascending document ID.
Rank 1 | Document 0 | Score: 0.000
Transformers use self-attention to understand relationships between words.

Query: 'document representation' | k=3
No query terms match the fitted vocabulary. All scores are zero; ties use ascending document ID.
Rank 1 | Document 0 | Score: 0.000
Transformers use self-attention to understand relationships between words.
Rank 2 | Document 1 | Score: 0.000
TF-IDF represents documents using word importance scores.
Rank 3 | Document 2 | Score: 0.000
Word2Vec learns dense vector representations of words.

Number of results: 1 versus 3


### Exercise 2 Answer

`k=1` returns document 0; `k=3` returns documents 0, 1 and 2. All scores are zero because the query’s singular terms do not match “documents” or “representations.” These are ties ordered by document ID, not relevant matches; changing k only changes how many results are returned.

## Verification

In [8]:
assert document_vectors.shape[0] == 6
assert document_vectors.shape[1] == len(vectorizer.get_feature_names_out())
assert np.isfinite(document_vectors.data).all()
assert np.allclose(np.asarray(document_vectors.multiply(document_vectors).sum(axis=1)).ravel(), 1)
assert scores.argmax() == 0
assert ner_results[0]['document_id'] == 3 and ner_results[0]['score'] > 0
assert len(results_k1) == 1 and len(results_k3) == 3
assert results_k1[0] == results_k3[0]
assert vectorizer.transform([representation_query]).nnz == 0
assert all(result['score'] == 0.0 for result in results_k3)
assert [result['document_id'] for result in results_k3] == [0, 1, 2]
assert all(ner_results[i]['score'] >= ner_results[i + 1]['score'] for i in range(len(ner_results) - 1))
print('All vectorization, ranking, exercise and zero-match checks passed.')

All vectorization, ranking, exercise and zero-match checks passed.
